# Generation and pairwise file construction

Attach the training notebook outputs as an input dataset. Every model is decoded with the same settings.

In [ ]:
REPO_URL = 'https://github.com/your-username/cs5326-pa2'
TRAIN_OUTPUTS = '/kaggle/input/cs5326-pa2-training/outputs'
DECODING = '--max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0 --no_repeat_ngram_size 0'

In [ ]:
!git clone {REPO_URL} /kaggle/working/cs5326-pa2

In [ ]:
%cd /kaggle/working/cs5326-pa2

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

## Task 2: Fixed AlpacaEval subset

In [ ]:
!python scripts/make_alpaca_subset.py --output_path eval_sets/alpaca_eval_300_seed42.json --size 300 --seed 42

## Task 6: Sanity check on three prompts

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/sft/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/sanity/sft_alpaca.json --generator_name sft --limit 3 --print_outputs {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/dpo_uf/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/sanity/dpo_uf_alpaca.json --generator_name dpo_uf --limit 3 --print_outputs {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/dpo_hh/best --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/sanity/dpo_hh.json --generator_name dpo_hh --limit 3 --print_outputs {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/lora_dpo/merged --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/sanity/lora_persona.json --generator_name lora_dpo --dataset_name persona_style_eval --prompt_format chat --limit 3 --print_outputs {DECODING}

## Task 2 and Task 6: Full generation on AlpacaEval

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/base_alpaca.json --generator_name qwen2.5-0.5b-base --dataset_name alpaca_eval {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/sft/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/sft_alpaca.json --generator_name sft --dataset_name alpaca_eval {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/dpo_uf/best --prompts_path eval_sets/alpaca_eval_300_seed42.json --output_path outputs/generations/dpo_uf_alpaca.json --generator_name dpo_uf --dataset_name alpaca_eval {DECODING}

## Generation on the synthetic helpful/harmless set

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/sft/best --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/sft_hh.json --generator_name sft --dataset_name hh_rlhf_style_eval {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/dpo_hh/best --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/dpo_hh_hh.json --generator_name dpo_hh --dataset_name hh_rlhf_style_eval {DECODING}

In [ ]:
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --prompts_path eval_sets/hh_rlhf_eval_300_seed42.json --output_path outputs/generations/base_hh.json --generator_name qwen2.5-0.5b-base --dataset_name hh_rlhf_style_eval {DECODING}

## Generation for the LoRA persona model

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/generations/base_persona.json --generator_name smollm2-360m-instruct --dataset_name persona_style_eval --prompt_format chat {DECODING}

In [ ]:
!python scripts/generate.py --model_path {TRAIN_OUTPUTS}/lora_dpo/merged --prompts_path eval_sets/persona_style_eval_40.json --output_path outputs/generations/lora_persona.json --generator_name lora_dpo --dataset_name persona_style_eval --prompt_format chat {DECODING}

## Randomized A/B pairwise files

In [ ]:
!python scripts/build_pairs.py --candidate_path outputs/generations/dpo_uf_alpaca.json --baseline_path outputs/generations/sft_alpaca.json --output_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --candidate_name dpo_uf --baseline_name sft --seed 42
!python scripts/build_pairs.py --candidate_path outputs/generations/dpo_hh_hh.json --baseline_path outputs/generations/sft_hh.json --output_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl --candidate_name dpo_hh --baseline_name sft --seed 42
!python scripts/build_pairs.py --candidate_path outputs/generations/lora_persona.json --baseline_path outputs/generations/base_persona.json --output_path outputs/eval/lora_vs_base_pairs.jsonl --candidate_name lora_dpo --baseline_name smollm2-instruct --seed 42

In [ ]:
!zip -qr /kaggle/working/generation_outputs.zip outputs/generations outputs/eval outputs/sanity eval_sets